# Colab L4 VRPO 500 campaign

每个 runtime 只启动一个静态分配的 job。只有 Drive 中的 readiness.json 已通过全部门禁且 execution SHA 一致时才会训练。当前计划元数据的 training_ready=false 不能用于启动。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import hashlib, json, os, subprocess, sys, tarfile
CAMPAIGN = Path('/content/drive/MyDrive/doudizhu/l4-vrpo-500-20261018-v1')
identity = json.loads((CAMPAIGN/'campaign_identity.json').read_text())
assert identity['parent_folder_id'] == '14wacgMnsJwnldEKydfQ5TFCUb5EOGTEO'
CANDIDATE, SEED = 'A0', 17


In [ ]:
execution = json.loads((CAMPAIGN/'execution.json').read_text())
ready = json.loads((CAMPAIGN/'readiness.json').read_text())
assert ready['training_ready'] is True
assert ready['execution_sha'] == execution['execution_sha']
def sha256(path):
    h = hashlib.sha256()
    with path.open('rb') as stream:
        for chunk in iter(lambda: stream.read(4*1024*1024), b''):
            h.update(chunk)
    return h.hexdigest()
for name, expected in execution['artifacts'].items():
    assert sha256(CAMPAIGN/'code'/name) == expected['sha256'], name
REPO = Path('/content')/('doudizhu-l4-'+execution['execution_sha'][:12])
if not REPO.exists():
    subprocess.run(['git','clone',str(CAMPAIGN/'code/execution.bundle'),str(REPO)],check=True)
subprocess.run(['git','checkout','--detach',execution['execution_sha']],cwd=REPO,check=True)
assert subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip() == ready['execution_sha']
with tarfile.open(CAMPAIGN/'code/native-runtime.tar.gz','r:gz') as archive:
    archive.extractall(REPO,filter='data')
subprocess.run([sys.executable,'-m','pip','install','-r',str(CAMPAIGN/'code/requirements.lock')],check=True)
subprocess.run([sys.executable,'-m','pip','install','--no-deps','-e',str(REPO)],check=True)


In [ ]:
ASSIGNMENT = CAMPAIGN/'assignments'/f'l4-vrpo500-{CANDIDATE}-s{SEED}.json'
env = {**os.environ, 'OMP_NUM_THREADS':'2', 'MKL_NUM_THREADS':'2', 'WANDB_DISABLED':'true', 'PYTHONHASHSEED':str(SEED), 'PYTHONPATH':str(REPO/'nanopokerlib/python')+':'+str(REPO), 'PYTHONUNBUFFERED':'1'}
command = [sys.executable,'-m','experiments.colab_l4_500.campaign','train','--candidate',CANDIDATE,'--seed',str(SEED),'--root','/content/l4-work','--ready',str(CAMPAIGN/'readiness.json'),'--drive-root',str(CAMPAIGN),'--assignment',str(ASSIGNMENT)]
print(command)
subprocess.run(command,cwd=REPO,env=env,check=True)


续训必须先从指定 Drive generation 独立下载并验证完整 archive，解包并放入对应 COMPLETE.json，再使用 campaign resume --resume <verified-directory> --target 501 做新 runtime 验收。通过后 target 才可设为 40000。该入口读取全部保存配置并校验候选，不能只加 --resume-from。